# State와 Node, Graph
* LangGraph를 구성하는 핵심 구성요소를 이해합니다

---
* 새로운 Conda 환경에서 실습을 진행하겠습니다.

  아래 명령어로 Python 3.12 환경(day15)을 생성한 후, 해당 환경을 활성화하고 ipykernel을 설치합니다
  
  ```
  conda create -n day15 python=3.12
  conda activate day15
  conda install ipykernel
  ```
  
  이렇게 하면 Jupyter Notebook에서 'day15' 환경을 커널로 사용할 수 있습니다.

In [1]:
!uv add python-dotenv langgraph pydantic

Resolved 396 packages in 9ms
Checked 360 packages in 96ms


In [2]:
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
OPENAI_API_KEY = __import__('os').getenv('OPENAI_API_KEY')

WORKDIR = Path.cwd()
ROOT = WORKDIR.parent

print('WORKDIR :', WORKDIR)

WORKDIR : d:\autornd\SK Autonomous R&D\실습\15일차_실습


---
## State 정의

LangGraph의 **State** = 그래프 전체가 공유하는 메모리.  
`State`는 대표적으로 `TypedDict`와 `BaseModel`을 사용해 정의합니다

## `TypedDict`
파이썬의 딕셔너리(`{}`)가 특정한 형식을 갖추도록 안내하는 방법

* 개발자 툴(IDE) 등에 표시만 해 줄 뿐, 오류를 일으키는 등 형식을 강제하지는 않습니다
* 가벼운 개인 프로젝트에 적합하고 여러 명이 개발하는 복잡한 상황에서는 추천하지 않습니다

In [3]:
from typing_extensions import TypedDict
from datetime import date

class Student(TypedDict):
    name: str
    student_number: int
    birth: date

In [4]:
student_a: Student = {
    'name': '홍길동',
    'student_number': 2026123123,
    'birth': date(2003, 10, 25)
}

student_a['birth']

datetime.date(2003, 10, 25)

In [5]:
student_b: Student = {
    'name': 123,
    'student_number': '홍길동'
} # 오류 발생 안함

student_b['name']

123

# `pydantic` `BaseModel`
* pydantic의 `BaseModel`을 활용하면 `TypedDict`보다 강력하게 형식을 강제할 수 있습니다
* `TypedDict`보다 연산이 조금 더 많지만 다양한 상황에 활용 가능합니다

In [6]:
from pydantic import BaseModel

class Student(BaseModel):
    name: str
    student_number: int
    birth: date

student_c = Student(name='홍길순', student_number=2026321321, birth=date(2002, 1, 1))

student_c.birth

datetime.date(2002, 1, 1)

In [7]:
student_d = Student(name='Jack', student_number=123, birth=date(2000, 1, 1))
student_d

Student(name='Jack', student_number=123, birth=datetime.date(2000, 1, 1))

## `BaseModel` 추가 활용법

### 1. 어떤 필드가 없어도 오류가 발생하지 않으려면 `Optional`을 사용합니다

In [8]:
from typing import Optional

class Players(BaseModel):
    player1: str
    player2: Optional[str] = None # str을 입력받지 않으면, None으로 취급

players = Players(player1= '홍길동')

players.player1

'홍길동'

In [9]:
players.player2 = '홍길순'
players

Players(player1='홍길동', player2='홍길순')

In [10]:
class Players(BaseModel):
    player1: str
    player2: Optional[str] = 'AI' # str을 입력받지 않으면, 'AI'로 취급

players = Players(player1= '홍길동')

players.player2

'AI'

In [11]:
players = Players(player1= '홍길동', player2='핫산') # 오류 발생
players

Players(player1='홍길동', player2='핫산')

### 2. 어떤 필드의 타입을 여러가지로 지정하려면 `|`을 활용합니다
* Any 를 사용하면 모든 타입을 허용할 수도 있습니다.

In [12]:
from typing import Any

class Anything(BaseModel):
    something: int | float | str | None = None
    anything: Optional[Any] = None

a = Anything(something=1)
b = Anything(something=0.5)
c = Anything(something='안녕하세요')
d = Anything(anything = date(1999, 12, 31))

for i in (a, b, c, d):
    print(i.something)

print(d.anything)

1
0.5
안녕하세요
None
1999-12-31


### 3. `Field` 를 사용해 더 강력한 제약을 넣거나 정보를 추가할 수도 있습니다.
* `ge`, `le`: 최소/최대값
* `description`: 해당 값에 대한 설명을 붙일 수 있음
* `pattern`: 값이 특정 형태를 따르도록 강제

In [13]:
from pydantic import Field

class SensorInput(BaseModel):
    temperature: int = Field(ge=0, le=100, description='센서 온도(℃)')
    humidity: int = Field(ge=0, le=100)
    operator: str = Field(pattern=r'^Operator\d$')

new_input = SensorInput(temperature=30, humidity=50, operator='Operator3')

new_input

SensorInput(temperature=30, humidity=50, operator='Operator3')

In [14]:
wrong_input = SensorInput(temperature=50, humidity=90, operator='Operator6') # 오류 발생

### 4. `Literal` 을 사용하면 특정 값을 강제할 수도 있습니다

In [15]:
from typing import Literal

class Person(BaseModel):
    name: str
    sex: Literal['여', '남', '여성']

a = Person(name='홍길동', sex='남')

In [16]:
b = Person(name='홍길순', sex='여성') # 오류 발생

### 5. `try`-`except`문을 활용하면 `ValidationError`를 구조화해 받을 수 있습니다

In [17]:
from pydantic import ValidationError

class SensorInput(BaseModel):
    temperature: int = Field(ge=0, le=100, description='센서 온도(℃)')

try:
    new_input = SensorInput(temperature=-30)
except ValidationError as e:
    print(e.errors())

[{'type': 'greater_than_equal', 'loc': ('temperature',), 'msg': 'Input should be greater than or equal to 0', 'input': -30, 'ctx': {'ge': 0}, 'url': 'https://errors.pydantic.dev/2.13/v/greater_than_equal'}]


In [18]:
while True:
    user_input = input("온도 값 입력: ").lower()
    if user_input in ('exit', '종료'):
        break
    try:
        new_input = SensorInput(temperature=user_input)
    except ValidationError as e:
        errors = e.errors()[0]
        key = list( errors['ctx'])[0]
        print(f"오류: 입력하신 값 '{errors['input']}'은 {'최소' if key == 'ge' else '최대'}값 {errors['ctx'][key]} 조건에 어긋납니다.")
        continue

    print("훌륭합니다")
    break

KeyboardInterrupt: Interrupted by user

## `State` 정의하기
* LangGraph 에서는 `State`를 중심으로 동작이 수행됩니다.
* 그래프를 구성하는 모든 노드는 `State`를 통해 정보를 주고받습니다.
* Langchain에서 멀티턴 대화를 위해 추가한 `history`도 `State`를 통해 관리합니다.

In [19]:
from typing import Literal

from pydantic import BaseModel, Field


class ChatMessage(BaseModel):
    role: Literal['user', 'assistant', 'system']
    content: str

class AgentState(BaseModel):
    # ── 대화 ──
    chat_history: list[ChatMessage] = Field(default_factory=list) # 맨 처음 생성 시 빈 리스트로 시작
    question: str = ''  # 이번 턴 사용자 입력
    tool_output: Optional[str] = None # 툴 사용 결과

---
## Node 함수 작성하기

* Node 함수는 일반적인 python 함수와 동일한 방식으로 작성합니다
* 모든 Node 함수는 `State`를 인자로 받고, `State`의 변경사항을 Dictionary로 출력합니다.

In [20]:
def question_node(state: AgentState) -> dict:
    user_question = state.question
    ai_response = f"입력하신 내용은 {user_question} 입니다."

    return {
        'chat_history': state.chat_history + [
            ChatMessage(role='user', content=user_question),
            ChatMessage(role='assistant', content=ai_response)
            ]
        }

In [21]:
def tool_node(state: AgentState) -> dict:
    return {
        'tool_output': "오늘 날씨는 맑습니다."
    }

# Node 연결해 그래프 만들기
* 만든 노드를 `edge`로 연결해 그래프를 만들 수 있습니다.
* `Graph` 객체에 `node`와 `edge`를 추가하고, `compile`과정을 거치면 그래프를 실행할 수 있습니다.

In [22]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(AgentState)

workflow.add_node("question", question_node)
workflow.add_node("tool", tool_node)

workflow.add_edge(START, "question") # 시작하면 question 노드로 간 다음
workflow.add_edge("question", "tool") # tool 노드로 이동하고
workflow.add_edge("tool", END) #실행 종료

app = workflow.compile()

In [23]:
init_input = AgentState(question="오늘 날씨 어때?") # 초기 시점의 State를 생성하고

final_state = app.invoke(init_input) # Graph에 입력으로 넣어주기

final_state

{'chat_history': [ChatMessage(role='user', content='오늘 날씨 어때?'),
  ChatMessage(role='assistant', content='입력하신 내용은 오늘 날씨 어때? 입니다.')],
 'question': '오늘 날씨 어때?',
 'tool_output': '오늘 날씨는 맑습니다.'}

## `stream()` 사용하기
* `app.stream()` 함수를 사용하면 실시간으로 어떤 노드가 실행되고 값이 어떻게 변하는지 모니터링할 수 있습니다

In [24]:
init_input = AgentState(question="오늘 날씨 어때?") # 초기 시점의 State를 생성하고

for chunk in app.stream(init_input):
    print(chunk)

{'question': {'chat_history': [ChatMessage(role='user', content='오늘 날씨 어때?'), ChatMessage(role='assistant', content='입력하신 내용은 오늘 날씨 어때? 입니다.')]}}
{'tool': {'tool_output': '오늘 날씨는 맑습니다.'}}


In [28]:
from pydantic import BaseModel
from typing import List

class ClassState(BaseModel):
    student: List[str]
    boring_count: int
    last_speech: Optional[str]

In [29]:
import time

def speech(state: ClassState) -> dict:
    sp = "ㅎㅇ"
    print(sp)
    return {
        'last_speech': sp,
        'boring_count': state.boring_count + 1
    }

def rest(state: ClassState) -> dict:
    print("휴식 시간입니다")
    time.sleep(2)
    return {
        'boring_count': 0
    }

In [35]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(ClassState)

workflow.add_node("say", speech)
workflow.add_node("run", rest)

workflow.add_edge(START, "say")
workflow.add_edge("say", "run")
workflow.add_edge("run", END)

app = workflow.compile()